In [ ]:
"""
================================================================================
CREDIT, MONETARY POLICY, AND DEFAULT HETEROGENEITY IN BRAZILIAN MSEs (2012–2026)
Econometric and Machine Learning Pipeline with Out-of-Time (OOT) Validation
Author: Itaiguara de Oliveira Bezerra
================================================================================
Methodology:
  1. Microdata ingestion from Central Bank of Brazil (SCR/BCB) & macro series (SGS/BCB).
  2. Sectoral & regional aggregation across Brazilian states and macro-sectors.
  3. Endogenous distress threshold calibration via Markov Chain Persistence (P11 >= 0.70).
  4. Supervised model training (2013–2021) and Out-of-Time testing (2022–2026).
  5. Discriminative & probabilistic evaluation (ROC-AUC, PR-AUC, Brier Score).
  6. Global & sector-specific directional interpretability via Tree SHAP.
  7. Exporting all publication-ready figures (.png) and summary tables (.xlsx) in English.
================================================================================
"""

from typing import Dict, List, Tuple
import warnings
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import shap

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
except ImportError:
    HAS_GEOPANDAS = False

warnings.filterwarnings('ignore')

# Global aesthetic styling
plt.style.use(
    'seaborn-v0_8-whitegrid'
    if 'seaborn-v0_8-whitegrid' in plt.style.available
    else 'default'
)

SECTOR_COLORS = {
    'Commerce': '#e377c2',
    'Construction': '#2ca02c',
    'Manufacturing': '#1f77b4',
    'Services': '#ff7f0e',
}

SECTOR_TRANSLATION = {
    'Comercio': 'Commerce',
    'Construcao': 'Construction',
    'Ind_Transformacao': 'Manufacturing',
    'Servicos': 'Services'
}

REGIAO_PT_TO_EN = {
    'Norte': 'North',
    'Nordeste': 'Northeast',
    'Centro-Oeste': 'Center-West',
    'Sudeste': 'Southeast',
    'Sul': 'South'
}

REGIONS_MAPPING = {
    'AC': 'North', 'AP': 'North', 'AM': 'North', 'PA': 'North', 'RO': 'North', 'RR': 'North', 'TO': 'North',
    'AL': 'Northeast', 'BA': 'Northeast', 'CE': 'Northeast', 'MA': 'Northeast', 'PB': 'Northeast',
    'PE': 'Northeast', 'PI': 'Northeast', 'RN': 'Northeast', 'SE': 'Northeast',
    'DF': 'Center-West', 'GO': 'Center-West', 'MT': 'Center-West', 'MS': 'Center-West',
    'ES': 'Southeast', 'MG': 'Southeast', 'RJ': 'Southeast', 'SP': 'Southeast',
    'PR': 'South', 'RS': 'South', 'SC': 'South'
}

FEATURE_LABELS_EN = {
    'selic_media': 'Average Selic Rate',
    'selic_lag3': 'Selic (Lag 3m)',
    'selic_lag6': 'Selic (Lag 6m)',
    'juros_giro': 'Working Capital Rate',
    'juros_giro_lag3': 'Working Capital Rate (Lag 3m)',
    'juros_giro_lag6': 'Working Capital Rate (Lag 6m)',
    'spread_bancario': 'Banking Spread',
    'ticket_medio_mil': 'Average Loan Size (BRL Th)',
    'crescimento_carteira_interanual': 'YoY Portfolio Growth (%)',
    'crescimento_operacoes_interanual': 'YoY Operations Growth (%)',
    'setor_macro_Commerce': 'Sector: Commerce',
    'setor_macro_Construction': 'Sector: Construction',
    'setor_macro_Manufacturing': 'Sector: Manufacturing',
    'setor_macro_Services': 'Sector: Services'
}


def remove_spines(ax: plt.Axes = None) -> None:
    """Removes chart frame spines for clean publication visualization."""
    if ax is None:
        ax = plt.gca()
    for spine in ['top', 'right', 'left', 'bottom']:
        ax.spines[spine].set_visible(False)


def load_scr_data(file_path: str = 'painel_scr_mpe_2012_2026') -> pd.DataFrame:
    """Loads raw SCR microdata from Parquet or CSV and standardizes sector names."""
    try:
        df = pd.read_parquet(f'{file_path}.parquet')
    except Exception:
        df = pd.read_csv(f'{file_path}.csv', sep=';', decimal=',')
    
    date_col = 'data' if 'data' in df.columns else 'data_base'
    df['data'] = pd.to_datetime(df[date_col])
    if 'setor_macro' in df.columns:
        df['setor_macro'] = df['setor_macro'].map(lambda x: SECTOR_TRANSLATION.get(x, x))
    return df


def fetch_sgs_series(series_code: int, column_name: str) -> pd.DataFrame:
    """Queries time series from the Central Bank of Brazil (SGS/BCB) API."""
    url = f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{series_code}/dados?formato=json'
    try:
        response = requests.get(url, timeout=20)
        if response.status_code == 200:
            df = pd.DataFrame(response.json())
            df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')
            df[column_name] = df['valor'].astype(float)
            df['year_month'] = df['data'].dt.to_period('M')
            return df[['year_month', column_name]].drop_duplicates(subset=['year_month'])
    except Exception as e:
        print(f'[WARNING] Failed to query SGS series {series_code}: {e}')
    
    raise ConnectionError(f'Unable to retrieve SGS series {series_code}.')


def plot_descriptive_profile(df_panel: pd.DataFrame, output_path: str = 'descriptive_profile_mse_base.png') -> None:
    """Plots aggregate credit volume, sectoral default trajectories, and boxplot distributions."""
    fig = plt.figure(figsize=(15, 10))
    gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.28, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0:2])
    ax2 = fig.add_subplot(gs[0, 2:4])
    ax3 = fig.add_subplot(gs[1, 1:3])

    # 1. Active Portfolio Growth
    df_time = df_panel.groupby('data')['carteira_ativa'].sum() / 1e9
    ax1.plot(df_time.index, df_time.values, color='#1f77b4', lw=2.5)
    ax1.set_ylabel('BRL Billion', fontweight='bold')
    ax1.set_xlabel('Year', fontweight='bold')
    remove_spines(ax1)

    # 2. Sectoral Default Rate
    for sector in sorted(df_panel['setor_macro'].unique()):
        subset = df_panel[df_panel['setor_macro'] == sector]
        ax2.plot(
            subset['data'],
            subset['default_rate_pct'],
            lw=2.2,
            label=sector,
            color=SECTOR_COLORS.get(sector, '#333333'),
        )
    ax2.set_ylabel('Default Rate (>90 Days) %', fontweight='bold')
    ax2.set_xlabel('Year', fontweight='bold')
    ax2.legend(loc='best', frameon=False, fontsize=9.5)
    remove_spines(ax2)

    # 3. Sectoral Boxplot
    sns.boxplot(
        data=df_panel,
        x='setor_macro',
        y='default_rate_pct',
        hue='setor_macro',
        palette='Blues',
        ax=ax3,
        legend=False,
        width=0.55,
    )
    ax3.set_xlabel('Macro Sector', fontweight='bold')
    ax3.set_ylabel('Default Rate (%)', fontweight='bold')
    remove_spines(ax3)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'[OK] Figure exported: {output_path}')


def plot_regional_panel(df_raw: pd.DataFrame, output_path: str = 'regional_sectoral_map_mse.png') -> None:
    """Plots regional choropleth map alongside sectoral risk bars across Brazilian regions."""
    df_geo = df_raw.copy()

    # Standardization to English
    if 'regiao' in df_geo.columns:
        df_geo['regiao'] = df_geo['regiao'].map(lambda x: REGIAO_PT_TO_EN.get(x, x))
    elif 'uf' in df_geo.columns:
        df_geo['regiao'] = df_geo['uf'].map(REGIONS_MAPPING)

    if 'setor_macro' in df_geo.columns:
        df_geo['setor_macro'] = df_geo['setor_macro'].map(lambda x: SECTOR_TRANSLATION.get(x, x))

    reg_summary_total = (
        df_geo.groupby('regiao')
        .agg(total_credit=('carteira_ativa', 'sum'), total_overdue=('vencido_acima_90', 'sum'))
        .reset_index()
    )
    reg_summary_total['default_rate_pct'] = (
        reg_summary_total['total_overdue'] / reg_summary_total['total_credit']
    ) * 100

    reg_summary_sector = (
        df_geo.groupby(['regiao', 'setor_macro'])
        .agg(sector_credit=('carteira_ativa', 'sum'), sector_overdue=('vencido_acima_90', 'sum'))
        .reset_index()
    )
    reg_summary_sector['default_rate_pct'] = (
        reg_summary_sector['sector_overdue'] / reg_summary_sector['sector_credit']
    ) * 100

    # Export Appendix Table for Brazilian States
    state_summary = (
        df_geo.groupby(['regiao', 'uf', 'setor_macro'])
        .agg(credit=('carteira_ativa', 'sum'), overdue=('vencido_acima_90', 'sum'))
        .reset_index()
    )
    state_summary['sector_default_pct'] = (state_summary['overdue'] / state_summary['credit']) * 100

    state_total = (
        df_geo.groupby(['regiao', 'uf'])
        .agg(tot_credit=('carteira_ativa', 'sum'), tot_overdue=('vencido_acima_90', 'sum'))
        .reset_index()
    )
    state_total['state_mean_default_pct'] = (state_total['tot_overdue'] / state_total['tot_credit']) * 100

    idx_crit = state_summary.groupby('uf')['sector_default_pct'].idxmax()
    df_appendix = state_summary.loc[idx_crit, ['regiao', 'uf', 'setor_macro', 'sector_default_pct']].copy()
    df_appendix = df_appendix.merge(state_total[['uf', 'state_mean_default_pct']], on='uf')
    df_appendix.columns = ['Macro Region', 'State (UF)', 'Critical Sector (Highest Default)', 'Sectoral Default (%)', 'Statewide Mean (%)']
    df_appendix.to_excel('appendix_critical_sector_by_state.xlsx', index=False)
    print('[OK] Spreadsheet exported: appendix_critical_sector_by_state.xlsx')

    fig_map = plt.figure(figsize=(19, 11), dpi=300)
    gs_geo = gridspec.GridSpec(5, 2, figure=fig_map, width_ratios=[1.15, 0.85], wspace=0.25, hspace=0.40)
    ax_map = fig_map.add_subplot(gs_geo[:, 0])

    map_plotted = False
    if HAS_GEOPANDAS:
        try:
            url_geojson = 'https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson'
            resp_geo = requests.get(url_geojson, headers={'User-Agent': 'Mozilla/5.0'}, timeout=25)
            if resp_geo.status_code == 200:
                gdf_states = gpd.read_file(resp_geo.text)
                gdf_states['regiao'] = gdf_states['sigla'].map(REGIONS_MAPPING)
                gdf_regions = gdf_states.dissolve(by='regiao', as_index=False)
                gdf_final = gdf_regions.merge(reg_summary_total, on='regiao', how='left')

                gdf_final.plot(
                    column='default_rate_pct',
                    cmap='YlOrRd',
                    linewidth=1.2,
                    edgecolor='#4a4a4a',
                    legend=True,
                    legend_kwds={
                        'label': 'Aggregate MSE Default Rate (>90 Days) %',
                        'orientation': 'horizontal',
                        'shrink': 0.65,
                        'pad': 0.02,
                    },
                    ax=ax_map,
                )

                offset_coords = {
                    'North': (-56.0, -3.5),
                    'Northeast': (-40.5, -8.5),
                    'Center-West': (-54.0, -15.5),
                    'Southeast': (-43.5, -20.5),
                    'South': (-52.0, -28.0),
                }

                for _, r_row in gdf_final.iterrows():
                    reg_name = r_row['regiao']
                    rate_val = r_row['default_rate_pct']
                    x_c, y_c = offset_coords.get(reg_name, (r_row.geometry.centroid.x, r_row.geometry.centroid.y))
                    ax_map.annotate(
                        text=f'{reg_name}\n{rate_val:.2f}%',
                        xy=(x_c, y_c),
                        ha='center',
                        va='center',
                        fontsize=10.5,
                        fontweight='bold',
                        color='#111111',
                        bbox=dict(boxstyle='round,pad=0.35', fc='#ffffff', ec='#777777', lw=0.8, alpha=0.9),
                    )
                ax_map.set_axis_off()
                map_plotted = True
        except Exception as e_map:
            print(f'[WARNING] GeoPandas mapping note: {e_map}')

    if not map_plotted:
        df_alt = reg_summary_total.sort_values('default_rate_pct', ascending=True)
        bars_alt = ax_map.barh(df_alt['regiao'], df_alt['default_rate_pct'], color='#d62728', height=0.55)
        for b in bars_alt:
            w = b.get_width()
            ax_map.text(w + 0.05, b.get_y() + b.get_height() / 2, f'{w:.2f}%', va='center', ha='left', fontsize=10, fontweight='bold')
        ax_map.set_xlabel('Default Rate (>90 Days) %', fontweight='bold')
        remove_spines(ax_map)

    regions_order = ['North', 'Northeast', 'Center-West', 'Southeast', 'South']
    for i, reg_name in enumerate(regions_order):
        ax_sub = fig_map.add_subplot(gs_geo[i, 1])
        sub_sector = reg_summary_sector[reg_summary_sector['regiao'] == reg_name].sort_values('default_rate_pct', ascending=True)
        
        if len(sub_sector) > 0:
            bar_colors = [SECTOR_COLORS.get(s, '#1f77b4') for s in sub_sector['setor_macro']]
            bars_reg = ax_sub.barh(sub_sector['setor_macro'], sub_sector['default_rate_pct'], color=bar_colors, height=0.6)
            max_v = sub_sector['default_rate_pct'].max()
            for b_reg in bars_reg:
                w_reg = b_reg.get_width()
                ax_sub.text(w_reg + (max_v * 0.02 if max_v > 0 else 0.05), b_reg.get_y() + b_reg.get_height() / 2, f'{w_reg:.2f}%', va='center', ha='left', fontsize=8.5, fontweight='bold', color='#222222')
            ax_sub.set_xlim(0, max_v * 1.25 if max_v > 0 else 1.0)
        
        ax_sub.set_title(f'Sectors in {reg_name}', fontsize=9.5, fontweight='bold', pad=4)
        ax_sub.tick_params(axis='y', labelsize=8.5)
        ax_sub.tick_params(axis='x', labelsize=8)
        remove_spines(ax_sub)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'[OK] Figure exported: {output_path}')


def optimize_markov_threshold(df_sector_train: pd.DataFrame, min_p11: float = 0.70) -> Tuple[float, float, float]:
    """Calibrates optimal cutoff tau_i where discrete-time Markov chain satisfies P11 >= min_p11."""
    series = df_sector_train['default_rate_pct'].values
    percentiles = np.linspace(50, 85, 36)
    tau_candidates = np.percentile(series, percentiles)
    
    best_tau = None
    best_p11 = 0.0
    best_p00 = 0.0
    
    for tau in tau_candidates:
        states = (series >= tau).astype(int)
        counts = np.zeros((2, 2))
        for t in range(len(states) - 1):
            counts[states[t], states[t+1]] += 1
        
        row_sums = counts.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        P = counts / row_sums
        
        p00, p11 = P[0, 0], P[1, 1]
        
        if p11 >= min_p11 and p00 >= 0.70:
            if p11 > best_p11:
                best_p11 = p11
                best_p00 = p00
                best_tau = tau
                
    if best_tau is None:
        best_tau = float(np.median(series))
        best_p11 = 0.50
        best_p00 = 0.50
        
    return float(best_tau), float(best_p00), float(best_p11)


def train_and_evaluate_models(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    oot_prevalence: float
) -> Tuple[Dict, pd.DataFrame, GradientBoostingClassifier]:
    """Fits classification algorithms and evaluates discriminative metrics on OOT test data."""
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 1. Logistic Regression
    log_reg = LogisticRegression(random_state=42, max_iter=1000)
    log_reg.fit(X_train_scaled, y_train)
    y_pred_lr, y_prob_lr = log_reg.predict(X_test_scaled), log_reg.predict_proba(X_test_scaled)[:, 1]

    # 2. Decision Tree
    dt = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
    dt.fit(X_train, y_train)
    y_pred_dt, y_prob_dt = dt.predict(X_test), dt.predict_proba(X_test)[:, 1]

    # 3. Random Forest
    rf = RandomForestClassifier(n_estimators=150, max_depth=4, min_samples_leaf=5, random_state=42)
    rf.fit(X_train, y_train)
    y_pred_rf, y_prob_rf = rf.predict(X_test), rf.predict_proba(X_test)[:, 1]

    # 4. Gradient Boosting
    gbm = GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42)
    gbm.fit(X_train, y_train)
    y_pred_gbm, y_prob_gbm = gbm.predict(X_test), gbm.predict_proba(X_test)[:, 1]

    models_dict = {
        '1. Logistic Regression (Baseline)': (y_pred_lr, y_prob_lr),
        '2. Decision Tree': (y_pred_dt, y_prob_dt),
        '3. Random Forest': (y_pred_rf, y_prob_rf),
        '4. Gradient Boosting': (y_pred_gbm, y_prob_gbm),
    }

    # Confusion Matrices Plot
    fig, axes = plt.subplots(2, 2, figsize=(10, 8.5))
    plots_config = [
        ('1. Logistic Regression (Baseline)', y_pred_lr, 'Purples', axes[0, 0]),
        ('2. Decision Tree', y_pred_dt, 'Blues', axes[0, 1]),
        ('3. Random Forest', y_pred_rf, 'Oranges', axes[1, 0]),
        ('4. Gradient Boosting', y_pred_gbm, 'Greens', axes[1, 1]),
    ]

    for subtitle, pred, cmap, ax in plots_config:
        cm = confusion_matrix(y_test, pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax, cbar=False, annot_kws={'size': 13, 'weight': 'bold'})
        ax.set_title(subtitle, fontsize=10.5, fontweight='bold', pad=8)
        ax.set_xlabel('Predicted', fontweight='bold', fontsize=9.5)
        ax.set_ylabel('Actual', fontweight='bold', fontsize=9.5)
        ax.set_xticklabels(['Low (0)', 'High (1)'])
        ax.set_yticklabels(['Low (0)', 'High (1)'])

    fig.tight_layout()
    plt.savefig('oot_confusion_matrices.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('[OK] Figure exported: oot_confusion_matrices.png')

    # ROC Curve Plot
    plt.figure(figsize=(8.5, 6))
    model_colors = ['#9467bd', '#1f77b4', '#ff7f0e', '#2ca02c']
    for (name, (_, prob)), col in zip(models_dict.items(), model_colors):
        fpr, tpr, _ = roc_curve(y_test, prob)
        auc_val = roc_auc_score(y_test, prob)
        plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.4f})', color=col, lw=2.2)

    plt.plot([0, 1], [0, 1], color='gray', linestyle=':', lw=1.5, label='Random Classifier (AUC = 0.5000)')
    plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=10, fontweight='bold')
    plt.ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=10, fontweight='bold')
    plt.legend(loc='lower right', frameon=False, fontsize=9.5)
    remove_spines()
    plt.tight_layout()
    plt.savefig('oot_roc_curve.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('[OK] Figure exported: oot_roc_curve.png')

    # PR Curve Plot
    plt.figure(figsize=(8.5, 6))
    for (name, (_, prob)), col in zip(models_dict.items(), model_colors):
        precision, recall, _ = precision_recall_curve(y_test, prob)
        pr_auc_val = average_precision_score(y_test, prob)
        plt.plot(recall, precision, label=f'{name} (PR-AUC = {pr_auc_val:.4f})', color=col, lw=2.2)

    plt.axhline(y=oot_prevalence, color='gray', linestyle=':', lw=1.5, label=f'No-Skill Baseline (Prevalence = {oot_prevalence:.2%})')
    plt.xlabel('Recall (Sensitivity)', fontsize=10, fontweight='bold')
    plt.ylabel('Precision (Positive Predictive Value)', fontsize=10, fontweight='bold')
    plt.ylim([0.0, 1.05])
    plt.xlim([0.0, 1.0])
    plt.legend(loc='upper right', frameon=False, fontsize=9.5)
    remove_spines()
    plt.tight_layout()
    plt.savefig('oot_pr_curve.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('[OK] Figure exported: oot_pr_curve.png')

    # Consolidated Metrics Table
    metrics_summary = []
    for name, (pred, prob) in models_dict.items():
        acc = accuracy_score(y_test, pred)
        roc = roc_auc_score(y_test, prob)
        pr = average_precision_score(y_test, prob)
        brier = brier_score_loss(y_test, prob)
        metrics_summary.append({
            'Model': name,
            'Accuracy': f'{acc:.2%}',
            'ROC-AUC': f'{roc:.4f}',
            'PR-AUC': f'{pr:.4f}',
            'Brier Score': f'{brier:.4f}'
        })

    df_metrics = pd.DataFrame(metrics_summary)
    df_metrics.to_excel('oot_models_performance_table.xlsx', index=False)
    print('[OK] Spreadsheet exported: oot_models_performance_table.xlsx')
    return models_dict, df_metrics, gbm


def compute_and_plot_shap(gbm_model: GradientBoostingClassifier, X_test: pd.DataFrame) -> None:
    """Computes directional SHAP values and generates global and sectoral visualizations."""
    explainer = shap.TreeExplainer(gbm_model)
    shap_values = explainer.shap_values(X_test)
    raw_feature_names = X_test.columns.tolist()
    display_feature_names = [FEATURE_LABELS_EN.get(f, f) for f in raw_feature_names]

    correlations_global = []
    for i, col in enumerate(raw_feature_names):
        feat_val = X_test[col].values
        s_val = shap_values[:, i]
        if np.std(feat_val) > 1e-6 and np.std(s_val) > 1e-6:
            corr = np.corrcoef(feat_val, s_val)[0, 1]
        else:
            corr = 0.0
        correlations_global.append(corr)

    mean_abs_shap_global = np.abs(shap_values).mean(axis=0)
    colors_global = ['#d62728' if c >= 0 else '#1f77b4' for c in correlations_global]

    df_shap_global = pd.DataFrame({
        'feature': display_feature_names,
        'importance': mean_abs_shap_global,
        'color': colors_global,
    }).sort_values('importance', ascending=True)

    patch_risk = mpatches.Patch(color='#d62728', label='Negative Impact (-): Positive Association with Risk')
    patch_protect = mpatches.Patch(color='#1f77b4', label='Positive Impact (+): Negative Association with Risk')

    # Global SHAP Bar Chart
    plt.figure(figsize=(11, 6), dpi=300)
    bars_global = plt.barh(
        df_shap_global['feature'],
        df_shap_global['importance'],
        color=df_shap_global['color'],
        edgecolor='black',
        alpha=0.85,
        height=0.70,
    )
    max_val_global = df_shap_global['importance'].max()
    for bar in bars_global:
        w = bar.get_width()
        plt.text(w + (max_val_global * 0.015), bar.get_y() + bar.get_height() / 2, f'{w:.2f}', va='center', ha='left', fontsize=9.5, fontweight='bold', color='#333333')

    plt.legend(handles=[patch_risk, patch_protect], loc='lower right', frameon=True, facecolor='white', edgecolor='none', fontsize=9.0)
    plt.xlim(0, max_val_global * 1.18)
    plt.xlabel('Mean Absolute Impact (|SHAP|)', fontsize=10, fontweight='bold')
    remove_spines()
    plt.tight_layout()
    plt.savefig('oot_global_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('[OK] Figure exported: oot_global_shap_summary.png')

    # Sectoral SHAP (2x2 Grid)
    idx_commerce = (X_test['setor_macro_Commerce'] == 1).values if 'setor_macro_Commerce' in X_test else np.zeros(len(X_test), dtype=bool)
    idx_construction = (X_test['setor_macro_Construction'] == 1).values if 'setor_macro_Construction' in X_test else np.zeros(len(X_test), dtype=bool)
    idx_manufacturing = (X_test['setor_macro_Manufacturing'] == 1).values if 'setor_macro_Manufacturing' in X_test else np.zeros(len(X_test), dtype=bool)
    idx_services = (X_test['setor_macro_Services'] == 1).values if 'setor_macro_Services' in X_test else np.zeros(len(X_test), dtype=bool)

    continuous_features = [f for f in raw_feature_names if not f.startswith('setor_macro_')]
    continuous_display_labels = [FEATURE_LABELS_EN.get(f, f) for f in continuous_features]
    feature_indices = [raw_feature_names.index(f) for f in continuous_features]

    fig_shap, axes_shap = plt.subplots(2, 2, figsize=(17, 11), dpi=300)
    axes_shap = axes_shap.flatten()

    sector_configs = [
        (axes_shap[0], 'SHAP: Commerce', idx_commerce),
        (axes_shap[1], 'SHAP: Construction', idx_construction),
        (axes_shap[2], 'SHAP: Manufacturing', idx_manufacturing),
        (axes_shap[3], 'SHAP: Services', idx_services),
    ]

    for ax, subtitle, sector_mask in sector_configs:
        if sector_mask.sum() > 0:
            shap_sub = shap_values[sector_mask, :]
            mean_shap_sector = np.abs(shap_sub[:, feature_indices]).mean(axis=0)

            sector_colors = []
            for f_idx, col in zip(feature_indices, continuous_features):
                val_f = X_test.loc[sector_mask, col].values
                val_s = shap_sub[:, f_idx]
                corr = np.corrcoef(val_f, val_s)[0, 1] if (np.std(val_f) > 1e-6 and np.std(val_s) > 1e-6) else 0.0
                sector_colors.append('#d62728' if corr >= 0 else '#1f77b4')
        else:
            mean_shap_sector = np.zeros(len(continuous_features))
            sector_colors = ['#1f77b4'] * len(continuous_features)

        df_sector_shap = pd.DataFrame({
            'feature': continuous_display_labels,
            'importance': mean_shap_sector,
            'color': sector_colors,
        }).sort_values('importance', ascending=True)

        bars = ax.barh(df_sector_shap['feature'], df_sector_shap['importance'], color=df_sector_shap['color'], edgecolor='black', alpha=0.85, height=0.68)
        max_v = df_sector_shap['importance'].max() if df_sector_shap['importance'].max() > 0 else 1.0
        for bar in bars:
            w = bar.get_width()
            ax.text(w + (max_v * 0.025), bar.get_y() + bar.get_height() / 2, f'{w:.1f}', va='center', ha='left', fontsize=9.0, fontweight='bold', color='#333333')

        ax.set_xlim(0, max_v * 1.35)
        ax.set_title(subtitle, fontsize=10.5, fontweight='bold', pad=8)
        ax.set_xlabel('Mean Absolute Impact (|SHAP|)', fontsize=9.0, fontweight='bold')
        ax.legend(handles=[patch_risk, patch_protect], loc='lower right', frameon=True, facecolor='white', edgecolor='none', fontsize=8.0)
        remove_spines(ax)

    plt.tight_layout()
    plt.savefig('oot_sector_shap_summary.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('[OK] Figure exported: oot_sector_shap_summary.png')


def main() -> None:
    print('=' * 80)
    print('ECONOMETRIC & MACHINE LEARNING PIPELINE: MSE DEFAULT RISK (2012–2026)')
    print('=' * 80)

    # 1. Ingestion and aggregation
    print('\n[1/5] Ingesting SCR microdata and consolidating sectors...')
    df_raw = load_scr_data()
    df_panel = (
        df_raw.groupby(['data', 'setor_macro'])
        .agg(
            carteira_ativa=('carteira_ativa', 'sum'),
            vencido_acima_90=('vencido_acima_90', 'sum'),
            numero_de_operacoes=('numero_de_operacoes', 'sum'),
        )
        .reset_index()
    )
    df_panel['default_rate_pct'] = np.where(
        df_panel['carteira_ativa'] > 0,
        (df_panel['vencido_acima_90'] / df_panel['carteira_ativa']) * 100,
        0.0,
    )
    df_panel['year_month'] = df_panel['data'].dt.to_period('M')

    # 2. Descriptive and regional diagnostics
    print('\n[2/5] Generating historical profile and regional risk map...')
    plot_descriptive_profile(df_panel)
    plot_regional_panel(df_raw)

    # 3. Covariates and macro-financial series
    print('\n[3/5] Structuring temporal lags and querying SGS/BCB API...')
    df_panel = df_panel.sort_values(['setor_macro', 'data']).reset_index(drop=True)
    df_panel['ticket_medio_mil'] = (df_panel['carteira_ativa'] / df_panel['numero_de_operacoes']) / 1000.0
    df_panel['crescimento_carteira_interanual'] = df_panel.groupby('setor_macro')['carteira_ativa'].pct_change(12) * 100
    df_panel['crescimento_operacoes_interanual'] = df_panel.groupby('setor_macro')['numero_de_operacoes'].pct_change(12) * 100

    df_selic = fetch_sgs_series(4189, 'selic_media')
    df_juros = fetch_sgs_series(20725, 'juros_giro')
    df_macro = pd.merge(df_selic, df_juros, on='year_month', how='inner').sort_values('year_month').reset_index(drop=True)
    df_macro['spread_bancario'] = df_macro['juros_giro'] - df_macro['selic_media']
    df_macro['selic_lag3'] = df_macro['selic_media'].shift(3)
    df_macro['selic_lag6'] = df_macro['selic_media'].shift(6)
    df_macro['juros_giro_lag3'] = df_macro['juros_giro'].shift(3)
    df_macro['juros_giro_lag6'] = df_macro['juros_giro'].shift(6)

    df_final = pd.merge(df_panel, df_macro, on='year_month', how='inner').dropna().reset_index(drop=True)

    # 4. Markov Threshold Optimization and OOT Split
    print('\n[4/5] Optimizing endogenous thresholds via Markov Persistence (P11 >= 0.70)...')
    temporal_split = pd.to_datetime('2022-01-01')
    df_train_raw = df_final[df_final['data'] < temporal_split].copy()
    df_test_raw = df_final[df_final['data'] >= temporal_split].copy()

    markov_thresholds = {}
    for sector in sorted(df_train_raw['setor_macro'].unique()):
        sub_tr = df_train_raw[df_train_raw['setor_macro'] == sector]
        tau_opt, p00, p11 = optimize_markov_threshold(sub_tr, min_p11=0.70)
        markov_thresholds[sector] = tau_opt
        print(f'  -> {sector:<20} | Threshold: {tau_opt:.2f}% | P00: {p00:.1%} | P11: {p11:.1%}')

    df_train_raw['high_default_target'] = (
        df_train_raw['default_rate_pct'] >= df_train_raw['setor_macro'].map(markov_thresholds)
    ).astype(int)
    df_test_raw['high_default_target'] = (
        df_test_raw['default_rate_pct'] >= df_test_raw['setor_macro'].map(markov_thresholds)
    ).astype(int)

    oot_prevalence = df_test_raw['high_default_target'].mean()
    n_positive = df_test_raw['high_default_target'].sum()
    print(f'Actual Distress Prevalence in OOT Test Set (2022–2026): {oot_prevalence:.2%} ({n_positive} of {len(df_test_raw)} months)')

    df_train = pd.get_dummies(df_train_raw, columns=['setor_macro'], drop_first=False, dtype=int)
    df_test = pd.get_dummies(df_test_raw, columns=['setor_macro'], drop_first=False, dtype=int)

    feature_cols = [
        'selic_media', 'selic_lag3', 'selic_lag6', 'juros_giro', 'juros_giro_lag3', 'juros_giro_lag6',
        'spread_bancario', 'ticket_medio_mil', 'crescimento_carteira_interanual', 'crescimento_operacoes_interanual'
    ] + [c for c in df_train.columns if c.startswith('setor_macro_')]

    X_train, y_train = df_train[feature_cols], df_train['high_default_target']
    X_test, y_test = df_test[feature_cols], df_test['high_default_target']

    # 5. Modeling and SHAP
    print('\n[5/5] Training supervised models and computing Tree SHAP...')
    _, df_metrics, gbm_model = train_and_evaluate_models(X_train, y_train, X_test, y_test, oot_prevalence)
    compute_and_plot_shap(gbm_model, X_test)

    print('\n' + '=' * 80)
    print('OUT-OF-TIME OFFICIAL EVALUATION METRICS (2022–2026):')
    print('=' * 80)
    print(df_metrics.to_string(index=False))
    print('\n[SUCCESS] Pipeline executed and all English artifacts generated!')


if __name__ == '__main__':
    main()